<a href="https://colab.research.google.com/github/jye110/ai-hw-spring-2026-nn/blob/master/MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# model.py
import torch.nn as nn
import torch.nn.functional as F
import torch
import math

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 28 -> 14
        x = self.pool(F.relu(self.conv2(x)))  # 14 -> 7
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)


class ShallowMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 256)
        self.dropout = nn.Dropout(0.25)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten the image
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x is (batch_size, seq_len, d_model) when batch_first=True
        return x + self.pe[:, :x.size(1), :]


class TransformerEncoderModel(nn.Module):
    def __init__(self, input_dim=28*28, embed_dim=128, num_heads=8, num_layers=1, dim_feedforward=256, dropout=0.1, num_classes=10):
        super().__init__()
        self.embed_dim = embed_dim
        # Embed each pixel value into embed_dim
        self.pixel_embedding = nn.Linear(1, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim, max_len=input_dim)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x shape: (batch_size, 1, 28, 28)
        batch_size = x.size(0)
        # Flatten to (batch_size, 28*28, 1) to treat each pixel as a sequence element with 1 feature
        x = x.view(batch_size, -1, 1)  # (batch_size, 784, 1)

        # Embed each pixel
        x = self.pixel_embedding(x)  # (batch_size, 784, embed_dim)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Pass through transformer encoder
        transformer_output = self.transformer_encoder(x)  # (batch_size, 784, embed_dim)

        # Global average pooling over the sequence dimension
        pooled_output = transformer_output.mean(dim=1)  # (batch_size, embed_dim)

        return self.classifier(pooled_output)

In [3]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_train_transform(aug_type):
    if aug_type == "none":
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "rotation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=10),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "translation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    if aug_type == "rotation+translation":
        return transforms.Compose([
            transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])


In [5]:
# train.py
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Assuming SimpleCNN, ShallowMLP, TransformerEncoderModel are defined in a prior cell
# and thus available in the global scope of the notebook.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the models to train
models_to_train = [
    ("CNN", SimpleCNN),
    ("MLP", ShallowMLP),
    ("Transformer", TransformerEncoderModel)
]

# Outer loop for different models
for model_name, ModelClass in models_to_train:
    print(f"\n--- Training {model_name} Model ---")
    for aug_type in ["none", "rotation", "translation", "rotation+translation"]:
        set_seed(42)

        transform_train = get_train_transform(aug_type)

        train_data = datasets.MNIST("./data", train=True, download=True, transform=transform_train)
        train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

        model = ModelClass().to(device) # Instantiate the current model
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)

        print(f"[{model_name} - {aug_type}] Starting training...")
        for epoch in range(5):
            model.train()
            total_loss = 0

            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                total_loss += loss.item()

            avg_loss = total_loss / len(train_loader)
            print(f"[{model_name} - {aug_type}] Epoch {epoch+1}/5 - Loss: {avg_loss:.4f}")

        transform_test = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize((0.1307,), (0.3081,))
      ])

        test_data = datasets.MNIST("./data", train=False, download=True, transform=transform_test)
        test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

        model.eval()

        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                predictions = outputs.argmax(dim=1)

                correct += (predictions == labels).sum().item()
                total += labels.size(0)

        accuracy = 100 * correct / total
        print(f"[{model_name} - {aug_type}] Final Test Accuracy: {accuracy:.2f}%\n")



--- Training CNN Model ---


100%|██████████| 9.91M/9.91M [00:00<00:00, 19.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 482kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.87MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.25MB/s]


[CNN - none] Starting training...
[CNN - none] Epoch 1/5 - Loss: 0.1629
[CNN - none] Epoch 2/5 - Loss: 0.0524
[CNN - none] Epoch 3/5 - Loss: 0.0384
[CNN - none] Epoch 4/5 - Loss: 0.0308
[CNN - none] Epoch 5/5 - Loss: 0.0243
[CNN - none] Final Test Accuracy: 99.12%

[CNN - rotation] Starting training...
[CNN - rotation] Epoch 1/5 - Loss: 0.1862
[CNN - rotation] Epoch 2/5 - Loss: 0.0686
[CNN - rotation] Epoch 3/5 - Loss: 0.0523
[CNN - rotation] Epoch 4/5 - Loss: 0.0426
[CNN - rotation] Epoch 5/5 - Loss: 0.0380
[CNN - rotation] Final Test Accuracy: 99.20%

[CNN - translation] Starting training...
[CNN - translation] Epoch 1/5 - Loss: 0.2754
[CNN - translation] Epoch 2/5 - Loss: 0.0991
[CNN - translation] Epoch 3/5 - Loss: 0.0759
[CNN - translation] Epoch 4/5 - Loss: 0.0637
[CNN - translation] Epoch 5/5 - Loss: 0.0563
[CNN - translation] Final Test Accuracy: 99.12%

[CNN - rotation+translation] Starting training...
[CNN - rotation+translation] Epoch 1/5 - Loss: 0.2993
[CNN - rotation+trans